# FND-1 Module 04: Data quality

Verify the immutable source, inspect grain before fields, separate seeded defects from accepted conditions, and make a bounded readiness decision. These records are synthetic and support no real clinical claim.

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
DATA = ROOT / 'data'
OUTPUTS = ROOT / 'outputs'
accepted = pd.read_csv(DATA / 'accepted-analytic-table.csv', keep_default_na=False)
defective = pd.read_csv(DATA / 'defective-analytic-table.csv', keep_default_na=False)
manifest = pd.read_csv(DATA / 'defect-manifest.csv', keep_default_na=False)
rules = pd.read_csv(OUTPUTS / 'quality-rule-results.csv', keep_default_na=False)
accepted.shape, defective.shape, manifest.shape, rules.shape

## 1. Verify source and grain

Rows are not automatically people. The accepted table must remain one row per synthetic patient, while the teaching layer intentionally contains five exact duplicates.

In [ ]:
source_path = DATA / 'accepted-analytic-table.csv'
source_sha = hashlib.sha256(source_path.read_bytes()).hexdigest()
grain = pd.DataFrame([
    {'layer': 'accepted', 'rows': len(accepted), 'people': accepted.patient_id.nunique(), 'duplicate_rows': int(accepted.duplicated().sum())},
    {'layer': 'defective', 'rows': len(defective), 'people': defective.patient_id.nunique(), 'duplicate_rows': int(defective.duplicated().sum())},
])
assert source_sha == '3c9944edc3806aa3b709a9ca08a9986a2f79978b1074ed098e31f19b533db25a'
assert grain.to_dict('records')[1]['duplicate_rows'] == 5
grain

## 2. Inspect missingness and rules

A blank is interpreted through its field contract. Optional death and next-event fields are not errors by default; blank required keys and required source values are seeded defects.

In [ ]:
missingness = pd.read_csv(OUTPUTS / 'missingness-profile.csv', keep_default_na=False)
quality = pd.read_csv(OUTPUTS / 'quality-profile.csv', keep_default_na=False)
assert len(missingness) == len(quality) == 29
assert len(rules) == 28 and (rules.detection_status == 'pass').all()
missingness.loc[missingness.delta_missing != 0, ['field_name', 'accepted_missing', 'defective_missing', 'delta_missing', 'structurally_allowed']]

## 3. Interpret risk and resolution

D-rules identify deliberate teaching defects. N-rules preserve accepted optionality, supported extremes, and small-cell cautions. Extreme is not the same as impossible, and rare is not the same as wrong.

In [ ]:
risk = pd.read_csv(OUTPUTS / 'quality-risk-log.csv', keep_default_na=False)
resolution = pd.read_csv(OUTPUTS / 'resolution-log.csv', keep_default_na=False)
resolved_path = OUTPUTS / 'resolved-analytic-table.csv'
resolved_sha = hashlib.sha256(resolved_path.read_bytes()).hexdigest()
assert set(risk.issue_id) == set(resolution.issue_id) == set(rules.issue_id)
assert resolved_sha == source_sha
resolution.groupby(['disposition', 'status']).size().rename('rules').reset_index()

## 4. Readiness decision

The initial defect layer requires **fix**. After deterministic restoration, the reference decision is **proceed with conditions**. Module 05 must use the exact resolved table, retain optional missingness, keep extreme-value and small-cell cautions visible, and make no real clinical or population claim from synthetic data.